[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch12-benchmarking/01-tap-kiem-tra-la-dung-cu-do.ipynb)

# Một điểm trên 1,000 ảnh: tập kiểm tra cũng là một dụng cụ đo

Phép tính nhẩm đầu tiên của chương 12 đặt một mô hình phân loại gốc đạt 95 phần trăm, còn bản nén
đạt 94 phần trăm, trên 1,000 ảnh, rồi hỏi: mức giảm một điểm ấy có chứng minh được một sự suy giảm
hay không? Câu trả lời của sách là chưa, vì tập kiểm tra là một dụng cụ đo, và dụng cụ 1,000 ảnh
không đủ mịn cho một thay đổi cỡ một điểm.

Sổ tay này làm lại phép tính ấy, rồi làm thí nghiệm với hai mô hình thật trên MNIST để thấy ba điều
chương nói mà không tính: so trên cùng các ảnh thì cần ít ảnh hơn hẳn, chọn mô hình tốt nhất trên
một tập kiểm tra cố định thì điểm số lạc quan hơn sự thật, và nhãn sai cũng là sai số của dụng cụ.

**Bạn sẽ làm:**
1. tính lại phép tính nhẩm 1.1: số lỗi, độ lệch chuẩn, khoảng tin cậy của mức chênh, và 1,825 mẫu;
2. tìm cỡ tập kiểm tra cần để phân giải mức chênh một điểm;
3. huấn luyện một mạng trên MNIST, nén nó bằng lượng tử hóa (quantization) trọng số làm bằng tay, rồi
   so hai mô hình bằng bootstrap và bằng kiểm định McNemar;
4. đo xem việc chọn mô hình thắng trên một tập kiểm tra cố định làm điểm số lạc quan tới đâu;
5. cấy 3 phần trăm nhãn sai vào tập kiểm tra, rồi tìm lại chúng.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

## 1. Phép tính nhẩm 1.1: một điểm trên 1,000 ảnh

Mỗi ảnh hoặc đúng hoặc sai, nên số lỗi trên $N$ ảnh theo phân phối nhị thức. Với độ chính xác thật
$p$, số lỗi có độ lệch chuẩn $\sqrt{N p (1-p)}$. Coi hai ước lượng $\hat p_1$ (bản gốc) và
$\hat p_2$ (bản nén) là độc lập, sách tính sai số chuẩn của hiệu giữa chúng:

$$\operatorname{SE}(\hat p_1-\hat p_2) = \sqrt{\frac{\hat p_1(1-\hat p_1)}{N} + \frac{\hat p_2(1-\hat p_2)}{N}}.$$

Khoảng tin cậy 95 phần trăm của mức chênh là mức chênh quan sát được, cộng trừ 1.96 lần sai số chuẩn.

**Dự đoán:** trước khi chạy, hãy đoán khoảng tin cậy của mức chênh một điểm rộng bao nhiêu điểm.

In [ ]:
from scipy import stats

N = sach(1_000, nguon="95 · 94 phần trăm · 1,000 ảnh")
p_goc = sach(95, nguon="95 · 94 phần trăm · 1,000 ảnh") / 100
p_nen = sach(94, nguon="95 · 94 phần trăm · 1,000 ảnh") / 100

loi_goc, loi_nen = N * (1 - p_goc), N * (1 - p_nen)
theo_sach("số lỗi của bản gốc", loi_goc, sach=50, nguon="50 · 60 lỗi")
theo_sach("số lỗi của bản nén", loi_nen, sach=60, nguon="50 · 60 lỗi")

do_lech_loi = math.sqrt(N * p_goc * (1 - p_goc))  # mô hình nhị thức của bản gốc
theo_sach("độ lệch chuẩn của số lỗi", do_lech_loi, sach=7, nguon="≈7 lỗi", tol=0.5)

Z95 = stats.norm.ppf(0.975)  # 1.96: nửa khoảng 95 % của phân phối chuẩn, tính bằng độ lệch chuẩn


def sai_so_chuan_hieu(pa, pb, n):
    """Sai số chuẩn của hiệu hai tỉ lệ đo trên hai tập n ảnh độc lập (công thức của sách)."""
    return math.sqrt(pa * (1 - pa) / n + pb * (1 - pb) / n)


se = sai_so_chuan_hieu(p_goc, p_nen, N)
chenh = 100 * (p_goc - p_nen)
duoi, tren = chenh - 100 * Z95 * se, chenh + 100 * Z95 * se
print(f"mức chênh {chenh:.1f} điểm, sai số chuẩn của hiệu {100 * se:.2f} điểm")
theo_sach("cận dưới của khoảng (điểm phần trăm)", duoi, sach=-1, nguon="−1 tới 3 điểm")
theo_sach("cận trên của khoảng (điểm phần trăm)", tren, sach=3, nguon="−1 tới 3 điểm")
assert duoi < 0 < tren  # khoảng chứa số không: chưa chứng minh được sự suy giảm

Khoảng chạy từ âm 1 tới 3 điểm và chứa số không. Mức chênh một điểm nằm gọn trong sai số của chính
phép đo, nên kết quả chưa chứng minh được bản nén kém hơn.

Con số 1,825 mẫu của sách trả lời một câu hỏi khác: cần bao nhiêu ảnh để đo **một** tỉ lệ cỡ 95
phần trăm với độ chính xác cộng trừ 1 điểm. Lời giải là $n = z^2 p(1-p) / \delta^2$, với nửa khoảng
$\delta$ bằng 1 điểm.

In [ ]:
nua_khoang = sach(1, nguon="1,825 mẫu · ±1 điểm") / 100
n_mot_ti_le = Z95 ** 2 * p_goc * (1 - p_goc) / nua_khoang ** 2
theo_sach("số mẫu để đo một tỉ lệ 95 % trong ±1 điểm", math.ceil(n_mot_ti_le), sach=1825, nguon="1,825 mẫu · ±1 điểm")

# Mức chênh giữa hai tỉ lệ có phương sai gần gấp đôi, nên cùng nửa khoảng 1 điểm cần nhiều ảnh hơn.
n_hieu = Z95 ** 2 * (p_goc * (1 - p_goc) + p_nen * (1 - p_nen)) / nua_khoang ** 2
# Giả định: muốn có 80 % cơ hội thấy mức chênh 1 điểm nếu nó có thật (lực kiểm định 80 %).
Z80 = stats.norm.ppf(0.80)
n_luc = (Z95 + Z80) ** 2 * (p_goc * (1 - p_goc) + p_nen * (1 - p_nen)) / nua_khoang ** 2
print(f"mỗi mô hình cần {math.ceil(n_hieu):,} ảnh để khoảng của mức chênh hẹp còn ±1 điểm")
print(f"và {math.ceil(n_luc):,} ảnh để có 80 % cơ hội phát hiện một mức chênh 1 điểm có thật")
assert n_hieu > 2 * n_mot_ti_le  # 1,825 mẫu không đủ cho phép so sánh

Hình dưới vẽ nửa khoảng tin cậy theo số ảnh, cho một tỉ lệ và cho mức chênh giữa hai tỉ lệ. Cả hai
hẹp lại theo $1/\sqrt{N}$: muốn khoảng hẹp đi một nửa thì phải có gấp bốn số ảnh.

In [ ]:
cac_n = np.logspace(2, 5, 200)
nua_mot = 100 * Z95 * np.sqrt(p_goc * (1 - p_goc) / cac_n)
nua_hieu = 100 * Z95 * np.sqrt((p_goc * (1 - p_goc) + p_nen * (1 - p_nen)) / cac_n)

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.loglog(cac_n, nua_mot, color="#2a78d6", lw=2, label="một tỉ lệ (95 %)")
ax.loglog(cac_n, nua_hieu, color="#eb6834", lw=2, label="mức chênh giữa hai mô hình độc lập")
ax.axhline(1, color="0.4", lw=1, ls=":")
ax.text(110, 1.06, "nửa khoảng 1 điểm", fontsize=8, color="0.3")
diem = [(N, nua_hieu[np.argmin(abs(cac_n - N))], f"{N:,} ảnh: ±{100 * Z95 * se:.1f} điểm", (10, 8)),
        (n_mot_ti_le, 1.0, "1,825 ảnh: ±1 điểm cho một tỉ lệ", (-150, -34)),
        (n_hieu, 1.0, f"{math.ceil(n_hieu):,} ảnh cho mức chênh", (12, 12))]
for x, y, chu, lech in diem:
    ax.plot(x, y, "o", color="black", ms=5)
    ax.annotate(chu, (x, y), xytext=lech, textcoords="offset points", fontsize=8,
                arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.set_yticks([0.2, 0.5, 1, 2, 5], ["0.2", "0.5", "1", "2", "5"])
ax.set_xticks([100, 1_000, 10_000, 100_000], ["100", "1,000", "10,000", "100,000"])
ax.set_xlabel("số ảnh của tập kiểm tra, N (thang log)")
ax.set_ylabel("nửa khoảng tin cậy 95 % (điểm phần trăm)")
ax.set_title("Tập kiểm tra càng nhỏ, dụng cụ đo càng thô")
ax.legend(loc="upper right", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

## 2. Hai mô hình thật trên cùng các ảnh

Công thức ở trên coi hai mô hình là độc lập, như thể mỗi mô hình được chấm trên một tập ảnh riêng.
Sách nhấn mạnh rằng nếu hai mô hình chạy trên **cùng** các ảnh thì phải giữ số ca hai mô hình bất
đồng và dùng một phép kiểm định ghép cặp, như kiểm định McNemar. Phần này làm đúng việc đó với hai
mô hình thật:

* **bản gốc:** mạng 784-128-64-10, huấn luyện 5 epoch trên 10,000 ảnh MNIST;
* **bản nén:** cùng mạng ấy, mỗi ma trận trọng số được lượng tử hóa xuống 3 bit, theo từng tensor,
  bằng tay. Sổ tay chọn 3 bit để mức giảm cỡ một điểm, như trong phép tính nhẩm.

Cả hai được chấm trên đủ 10,000 ảnh của tập kiểm tra MNIST.

In [ ]:
import copy

import torch.nn.functional as F
import torchvision

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # thứ tự màu cố định

tap_hl = torchvision.datasets.MNIST(DATA, train=True, download=True)
tap_kt = torchvision.datasets.MNIST(DATA, train=False, download=True)


def phang(anh_uint8):
    return anh_uint8.reshape(len(anh_uint8), 784).float() / 255


N_HL = 10_000  # tập con huấn luyện, để cả sổ tay chạy trong khoảng một phút
X_hl, y_hl = phang(tap_hl.data[:N_HL]), tap_hl.targets[:N_HL]
X_kt, y_kt = phang(tap_kt.data), tap_kt.targets
print(f"huấn luyện trên {len(X_hl):,} ảnh, kiểm tra trên {len(X_kt):,} ảnh")

dat_hat_giong()
mang = torch.nn.Sequential(torch.nn.Linear(784, 128), torch.nn.ReLU(),
                           torch.nn.Linear(128, 64), torch.nn.ReLU(),
                           torch.nn.Linear(64, 10))
toi_uu = torch.optim.Adam(mang.parameters(), lr=1e-3)
for epoch in range(5):
    thu_tu = torch.randperm(N_HL)
    for i in range(0, N_HL, 64):
        lo = thu_tu[i:i + 64]
        mat_mat = F.cross_entropy(mang(X_hl[lo]), y_hl[lo])
        toi_uu.zero_grad()
        mat_mat.backward()
        toi_uu.step()


def luong_tu_hoa(mo_hinh, bit):
    """Bản sao có mọi ma trận trọng số làm tròn về 2^bit mức đối xứng, một hệ số co giãn mỗi tensor."""
    ban = copy.deepcopy(mo_hinh)
    muc = 2 ** (bit - 1) - 1  # 3 bit: các số nguyên −3..3
    with torch.no_grad():
        for lop in ban:
            if isinstance(lop, torch.nn.Linear):
                he_so = lop.weight.abs().max() / muc
                lop.weight.copy_(torch.round(lop.weight / he_so).clamp(-muc, muc) * he_so)
    return ban


BIT = 3
mang_nen = luong_tu_hoa(mang, BIT)
assert len(torch.unique(mang_nen[0].weight)) <= 2 ** BIT  # đúng là chỉ còn 2^3 giá trị
with torch.no_grad():
    dung_goc = (mang(X_kt).argmax(1) == y_kt).numpy()
    dung_nen = (mang_nen(X_kt).argmax(1) == y_kt).numpy()
do_tren_may("độ chính xác của bản gốc trên 10,000 ảnh", 100 * dung_goc.mean(), "%")
do_tren_may(f"độ chính xác của bản nén {BIT} bit trên 10,000 ảnh", 100 * dung_nen.mean(), "%")

### Bảng bất đồng

Hai mô hình đúng sai trên phần lớn **cùng** các ảnh, vì bản nén là chính bản gốc với trọng số làm
tròn. Bảng dưới đếm bốn trường hợp. Chỉ hai ô giữa, nơi hai mô hình bất đồng, mang thông tin về
việc mô hình nào tốt hơn.

**Dự đoán:** trong 10,000 ảnh, hai mô hình bất đồng ở bao nhiêu ảnh: vài chục, vài trăm, hay vài
nghìn?

In [ ]:
def bang_bat_dong(a, b):
    return {"cả hai đúng": int(np.sum(a & b)), "chỉ bản gốc đúng": int(np.sum(a & ~b)),
            "chỉ bản nén đúng": int(np.sum(~a & b)), "cả hai sai": int(np.sum(~a & ~b))}


for ten, so in bang_bat_dong(dung_goc, dung_nen).items():
    print(f"{ten:>17}: {so:>6,}")
assert sum(bang_bat_dong(dung_goc, dung_nen).values()) == len(y_kt)


def mcnemar(a, b):
    """Kiểm định McNemar chính xác: dưới giả thuyết không, mỗi ca bất đồng nghiêng về bên nào là 50/50."""
    chi_a, chi_b = int(np.sum(a & ~b)), int(np.sum(~a & b))
    return chi_a, chi_b, stats.binomtest(chi_a, chi_a + chi_b, 0.5).pvalue

### Bootstrap: lấy lại mẫu từ chính tập kiểm tra

Bootstrap ước lượng khoảng tin cậy mà không cần công thức: lấy lại $n$ ảnh có hoàn lại từ tập kiểm
tra, tính mức chênh, lặp 2,000 lần, rồi đọc phân vị 2.5 và 97.5. Có hai cách lấy lại:

* **không ghép cặp:** mỗi mô hình được lấy lại ảnh riêng, như công thức của sách;
* **ghép cặp:** cùng một bộ ảnh cho cả hai mô hình, đúng như cách hai mô hình thực sự được chấm.

Sổ tay làm cả hai cách cho một benchmark 1,000 ảnh (1,000 ảnh đầu của tập kiểm tra) và cho đủ
10,000 ảnh.

In [ ]:
def khoang_bootstrap(a, b, ghep_cap, lap=2_000, seed=SEED):
    """Khoảng 95 % (điểm phần trăm) của độ chính xác a trừ độ chính xác b."""
    rng = np.random.default_rng(seed)
    n, hieu = len(a), np.empty(lap)
    for dau in range(0, lap, 250):
        k = min(250, lap - dau)
        chi_so_a = rng.integers(0, n, size=(k, n))
        chi_so_b = chi_so_a if ghep_cap else rng.integers(0, n, size=(k, n))
        hieu[dau:dau + k] = a[chi_so_a].mean(1) - b[chi_so_b].mean(1)
    return 100 * np.percentile(hieu, [2.5, 97.5])


ket_qua = []
for n in (1_000, 10_000):
    a, b = dung_goc[:n], dung_nen[:n]
    quan_sat = 100 * (a.mean() - b.mean())
    cong_thuc = 100 * Z95 * sai_so_chuan_hieu(a.mean(), b.mean(), n)
    chi_a, chi_b, p = mcnemar(a, b)
    for ghep in (False, True):
        lo_, hi_ = khoang_bootstrap(a, b, ghep)
        ket_qua.append((n, ghep, quan_sat, lo_, hi_))
        cach = "ghép cặp" if ghep else "không ghép cặp"
        do_tren_may(f"n = {n:,}, {cach}, cận dưới", round(lo_, 2), "điểm")
        do_tren_may(f"n = {n:,}, {cach}, cận trên", round(hi_, 2), "điểm")
    print(f"   công thức độc lập của sách cho n = {n:,}: {quan_sat:+.2f} ± {cong_thuc:.2f} điểm")
    do_tren_may(f"n = {n:,}, McNemar: {chi_a} ca chỉ bản gốc đúng, {chi_b} ca chỉ bản nén đúng, p", p)
    print("   khoảng ghép cặp", "CHỨA" if lo_ < 0 < hi_ else "KHÔNG chứa", "số không")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.6))
for i, (n, ghep, quan_sat, lo_, hi_) in enumerate(ket_qua):
    mau = MAU[1] if ghep else MAU[0]
    ax.plot([lo_, hi_], [i, i], color=mau, lw=5, solid_capstyle="butt")
    ax.plot(quan_sat, i, "o", color="black", ms=5)
    ax.text(hi_ + 0.08, i, f"{lo_:+.2f} … {hi_:+.2f}", va="center", fontsize=8)
ax.axvline(0, color="0.3", lw=1, ls=":")
ax.set_yticks(range(len(ket_qua)),
              [f"{n:,} ảnh, {'ghép cặp' if ghep else 'không ghép cặp'}" for n, ghep, *_ in ket_qua])
ax.invert_yaxis()
xs = [v for r in ket_qua for v in r[3:]]
ax.set_xlim(min(xs + [0]) - 0.3, max(xs) + 1.2)
ax.set_xlabel("độ chính xác bản gốc trừ bản nén (điểm phần trăm)")
ax.set_title("Khoảng tin cậy bootstrap 95 % của mức chênh (đo trên máy này)", fontsize=10)
plt.tight_layout()
plt.show()

Hai điều đọc được từ hình, dù con số cụ thể trên máy bạn khác đi:

* Khoảng không ghép cặp gần như trùng với công thức độc lập của sách, và hẹp đi khoảng
  $\sqrt{10} \approx 3.2$ lần khi tập kiểm tra lớn gấp mười.
* Khoảng ghép cặp hẹp hơn hẳn ở cùng số ảnh. Phần lớn ảnh được hai mô hình chấm giống nhau, và một
  ảnh cả hai cùng đúng hay cùng sai không đóng góp gì vào mức chênh. Phương sai chỉ đến từ các ca bất
  đồng, đúng như sách nói: chỉ so hai độ chính xác tổng là không đủ.

Từ đó tính được số ảnh mà một phép so sánh ghép cặp cần, với chính tỉ lệ bất đồng đo được:

In [ ]:
d = dung_goc.astype(float) - dung_nen.astype(float)  # +1, 0 hoặc −1 cho mỗi ảnh
n_ghep = Z95 ** 2 * d.var() / nua_khoang ** 2
do_tren_may("tỉ lệ ảnh hai mô hình bất đồng", 100 * np.mean(d != 0), "%")
do_tren_may("số ảnh để khoảng ghép cặp hẹp còn ±1 điểm", math.ceil(n_ghep), "ảnh")
print(f"so với {math.ceil(n_hieu):,} ảnh theo công thức độc lập ở phần 1")

## 3. Chọn mô hình thắng trên một tập kiểm tra cố định

Định luật Goodhart nói khi thước đo thành mục tiêu, nó thôi là thước đo tốt. Một dạng rất đời thường
của nó: thử nhiều mô hình, giữ mô hình có điểm cao nhất trên cùng một tập kiểm tra, rồi báo đúng điểm
ấy. Mỗi điểm số chứa một phần nhiễu đo, và mô hình thắng thường là mô hình gặp may nhất. Thiết kế chỉ
cho benchmark là quá khớp (overfitting), như sách viết.

Thí nghiệm dưới đây có 40 mô hình chỉ khác nhau ở seed. Mỗi mô hình chiếu ảnh qua 256 đặc trưng
ngẫu nhiên có ReLU, rồi giải một hồi quy ridge bằng công thức đóng. Chúng giỏi gần như nhau, nên mọi
khác biệt lớn trên một tập nhỏ phần nhiều là may rủi. Tập kiểm tra 10,000 ảnh được chia thành một tập
**công khai** 1,000 ảnh để chọn mô hình, và 9,000 ảnh **mới** mà mô hình được chọn chưa từng được
chấm trên đó.

**Dự đoán:** mô hình thắng trên tập công khai sẽ có điểm trên tập mới cao hơn, bằng, hay thấp hơn điểm
công khai của nó?

In [ ]:
K, H = 40, 256
Y_hl = F.one_hot(y_hl, 10).float()
du_doan = []
for k in range(K):
    g = torch.Generator().manual_seed(1_000 + k)
    W = 2 * torch.randn(784, H, generator=g) / math.sqrt(784)
    lech = 0.5 * torch.randn(H, generator=g)
    Phi = torch.relu(X_hl @ W + lech)
    beta = torch.linalg.solve(Phi.T @ Phi + torch.eye(H), Phi.T @ Y_hl)
    du_doan.append((torch.relu(X_kt @ W + lech) @ beta).argmax(1).numpy())
dung_k = np.stack(du_doan) == y_kt.numpy()  # (40 mô hình, 10,000 ảnh)
assert dung_k.shape == (K, len(y_kt))

diem_du = 100 * dung_k.mean(1)
do_tren_may("độ chính xác trung bình của 40 mô hình trên đủ 10,000 ảnh", diem_du.mean(), "%")
do_tren_may("chênh giữa mô hình tốt nhất và kém nhất trên đủ 10,000 ảnh", diem_du.max() - diem_du.min(), "điểm")

rng = np.random.default_rng(SEED)
thu_tu = rng.permutation(len(y_kt))
cong_khai, moi = thu_tu[:1_000], thu_tu[1_000:]
diem_ck, diem_moi = 100 * dung_k[:, cong_khai].mean(1), 100 * dung_k[:, moi].mean(1)
thang = int(np.argmax(diem_ck))
do_tren_may("mô hình thắng: điểm trên tập công khai", diem_ck[thang], "%")
do_tren_may("mô hình thắng: điểm trên 9,000 ảnh mới", diem_moi[thang], "%")
do_tren_may("trung bình 40 mô hình trên 9,000 ảnh mới", diem_moi.mean(), "%")

Một lần chia chỉ là một lần gieo xúc xắc. Vì đã có dự đoán của cả 40 mô hình trên mọi ảnh, sổ tay chia
lại tập kiểm tra 300 lần, mỗi lần chọn mô hình thắng trên phần công khai và chấm nó trên phần còn lại,
với số mô hình $K$ và cỡ tập công khai khác nhau. Độ lạc quan là điểm công khai của mô hình thắng trừ
điểm của nó trên ảnh mới.

Đường mô phỏng vẽ cùng phép chọn với một giả định đơn giản hơn: $K$ mô hình có độ chính xác thật bằng
nhau, bằng độ chính xác trung bình đo được của 40 mô hình, và sai **độc lập** với nhau.

In [ ]:
def do_lac_quan(dung, k, n_ck, lan=300, seed=SEED):
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(lan):
        mo_hinh = rng.choice(len(dung), size=k, replace=False)
        tt = rng.permutation(dung.shape[1])
        ck, con_lai = tt[:n_ck], tt[n_ck:]
        a = dung[mo_hinh][:, ck].mean(1)
        j = int(np.argmax(a))
        out.append(a[j] - dung[mo_hinh[j], con_lai].mean())
    return 100 * float(np.mean(out))


def mo_phong(k, n_ck, p, lan=20_000, seed=SEED):
    rng = np.random.default_rng(seed)
    return 100 * float(np.mean(rng.binomial(n_ck, p, size=(lan, k)).max(1) / n_ck - p))


cac_K = [1, 2, 5, 10, 20, 40]
lac_quan = {n_ck: [do_lac_quan(dung_k, k, n_ck) for k in cac_K] for n_ck in (1_000, 5_000)}
p_tb = float(dung_k.mean())  # giả định của mô phỏng: mọi mô hình giỏi bằng mức trung bình
mo_phong_1000 = [mo_phong(k, 1_000, p_tb) for k in cac_K]
for n_ck, ds in lac_quan.items():
    do_tren_may(f"công khai {n_ck:,} ảnh, chọn trong 40 mô hình: độ lạc quan trung bình", ds[-1], "điểm")
print(f"mô phỏng sai độc lập, công khai 1,000 ảnh, K = 40: {mo_phong_1000[-1]:.2f} điểm")
assert abs(mo_phong(1, 1_000, p_tb)) < 0.05  # không chọn thì không lạc quan

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.scatter(diem_ck, diem_moi, s=18, color=MAU[0], alpha=0.8, label="40 mô hình")
ax1.scatter(diem_ck[thang], diem_moi[thang], s=60, color=MAU[1], zorder=3, label="mô hình thắng")
lo_, hi_ = min(diem_ck.min(), diem_moi.min()) - 0.3, max(diem_ck.max(), diem_moi.max()) + 0.3
ax1.plot([lo_, hi_], [lo_, hi_], color="0.5", lw=1, ls=":")
ax1.text(hi_ - 0.1, hi_ - 0.35, "y = x", fontsize=8, color="0.4", ha="right")
ax1.set_xlim(lo_, hi_)
ax1.set_ylim(lo_, hi_)
ax1.set_xlabel("điểm trên 1,000 ảnh công khai (%)")
ax1.set_ylabel("điểm trên 9,000 ảnh mới (%)")
ax1.set_title("Một lần chia")
ax1.legend(loc="upper left", fontsize=8)
ax2.plot(cac_K, mo_phong_1000, "s--", color="0.5", lw=1.5, label="mô phỏng: sai độc lập, 1,000 ảnh")
for (n_ck, ds), mau in zip(lac_quan.items(), (MAU[1], MAU[2])):
    ax2.plot(cac_K, ds, "o-", color=mau, lw=2, label=f"đo: công khai {n_ck:,} ảnh")
ax2.set_xscale("log")
ax2.set_xticks(cac_K, [str(k) for k in cac_K])
ax2.set_xlabel("số mô hình được thử, K (thang log)")
ax2.set_ylabel("điểm công khai trừ điểm mới (điểm phần trăm)")
ax2.set_title("Độ lạc quan của mô hình thắng")
ax2.legend(loc="upper left", fontsize=8)
fig.suptitle("Chọn mô hình thắng trên một tập kiểm tra cố định (đo trên máy này)", fontsize=10)
plt.tight_layout()
plt.show()

Độ lạc quan tăng theo số mô hình được thử, và giảm khi tập công khai lớn hơn. Đường đo thường nằm
dưới đường mô phỏng: các mô hình thật sai trên nhiều ảnh chung, nên điểm của chúng không dao động
độc lập, và phần may rủi để khai thác nhỏ hơn. Nhưng nó không bằng không, và nó lớn dần đúng như
Goodhart dự báo.

Đây là lý do sách đòi một báo cáo phải nêu phân phối của các lần chạy được chấp nhận, thay vì chỉ
một lần chạy tốt nhất, và lý do MLPerf có quy tắc chạy chặt. Một bảng xếp hạng công khai mà ai cũng
được nộp lại nhiều lần là đúng thí nghiệm này, ở quy mô lớn hơn.

## 4. Nhãn sai cũng là sai số của dụng cụ

Sách dẫn các nghiên cứu thấy tỉ lệ lỗi nhãn 3 tới 6 phần trăm trong các tập lớn, kể cả ImageNet. Một
nhãn sai trong tập kiểm tra chấm sai mô hình, dù mô hình đúng. Phần này cấy 3 phần trăm nhãn sai vào
tập kiểm tra MNIST, mỗi nhãn bị đổi sang một chữ số khác chọn ngẫu nhiên, rồi xem hai điều: phép đo
lệch bao nhiêu, và mô hình có giúp tìm lại các nhãn ấy không.

Cách tìm là ý tưởng của học tự tin (confident learning) mà sách nhắc: tìm những ảnh mà dự đoán của mô
hình bất đồng một cách có hệ thống với nhãn. Ở đây, sắp các ảnh theo xác suất mô hình gán cho chính
nhãn của ảnh, rồi xem 300 ảnh có xác suất thấp nhất.

In [ ]:
ti_le_sai = sach(3, nguon="3–6 phần trăm") / 100
rng = np.random.default_rng(SEED)
nhan = y_kt.numpy().copy()
so_sai = int(round(ti_le_sai * len(nhan)))
bi_doi = rng.choice(len(nhan), size=so_sai, replace=False)
nhan[bi_doi] = (nhan[bi_doi] + rng.integers(1, 10, size=so_sai)) % 10  # luôn sang một chữ số khác
assert np.sum(nhan != y_kt.numpy()) == so_sai == 300

with torch.no_grad():
    xac_suat = torch.softmax(mang(X_kt), 1).numpy()
du_doan_goc = xac_suat.argmax(1)
do_tren_may("độ chính xác chấm theo nhãn đúng", 100 * np.mean(du_doan_goc == y_kt.numpy()), "%")
do_tren_may("độ chính xác chấm theo nhãn đã cấy lỗi", 100 * np.mean(du_doan_goc == nhan), "%")

tin_vao_nhan = xac_suat[np.arange(len(nhan)), nhan]
nghi_ngo = np.argsort(tin_vao_nhan)[:so_sai]
trung = np.isin(nghi_ngo, bi_doi)
do_tren_may("trong 300 ảnh bị nghi nhất, phần là nhãn đã cấy lỗi", 100 * trung.mean(), "%")
print(f"chọn ngẫu nhiên 300 ảnh thì chỉ trúng khoảng {100 * ti_le_sai:.0f} %")

Những ảnh bị nghi mà **không** nằm trong số nhãn đã cấy lỗi mang nhãn gốc của MNIST. Một số trong đó có
thể thật sự mơ hồ, số khác chỉ là lỗi của mô hình. Việc xem bằng mắt, như sách gợi ý kiểm tra thủ
công một mẫu, mới quyết định được.

In [ ]:
goc_nghi = [i for i in nghi_ngo if i not in set(bi_doi)][:10]
fig, axs = plt.subplots(1, 10, figsize=(11, 1.9))
for ax, i in zip(axs, goc_nghi):
    ax.imshow(tap_kt.data[i], cmap="gray_r")
    ax.set_title(f"nhãn {nhan[i]}\nmô hình {du_doan_goc[i]}", fontsize=8)
    ax.axis("off")
fig.suptitle("Mười ảnh có nhãn gốc mà mô hình nghi nhất (đo trên máy này)", fontsize=10)
plt.tight_layout()
plt.show()

Cách tìm này bắt được nhãn sai **ngẫu nhiên**, vì một mô hình học từ phần lớn dữ liệu đúng sẽ không đồng
ý với chúng. Sách cảnh báo một thiên lệch **nhất quán** thì khác: nếu mọi con sói chụp trên tuyết đều
bị gán là chó, mô hình học đúng quy tắc sai ấy, đồng ý với nhãn, và cách tìm này không thấy gì.

## Thử thêm

1. Ở phần 1, thêm dòng `N = 10_000` ngay dưới dòng `N = sach(...)` rồi chạy lại ô ấy (dấu ✗ khi đó là
   điều được chờ đợi, vì sách tính với 1,000 ảnh). Khoảng của mức chênh một điểm còn chứa số không không?
2. Ở phần 2, đổi `BIT` thành 4 rồi 2. Số ca bất đồng thay đổi ra sao, và khi nào McNemar thấy được
   mức chênh trên 1,000 ảnh?
3. Ở phần 3, đổi `H` thành 64 để các mô hình kém hơn và khác nhau nhiều hơn, hoặc đổi cỡ tập công khai
   trong `do_lac_quan` thành 200. Độ lạc quan đổi thế nào?

## Tóm lại

* Trên 1,000 ảnh, mức chênh giữa 95 và 94 phần trăm có khoảng tin cậy từ âm 1 tới 3 điểm, nên chưa
  chứng minh được sự suy giảm. Đo một tỉ lệ trong ±1 điểm cần 1,825 ảnh, đo một mức chênh cần hơn gấp đôi.
* Hai mô hình chấm trên cùng các ảnh phải được so ghép cặp: chỉ các ca bất đồng mang thông tin, và
  khoảng ghép cặp hẹp hơn hẳn khoảng tính từ hai độ chính xác tổng.
* Chọn mô hình thắng trên một tập kiểm tra cố định làm điểm số lạc quan, nhiều hơn khi thử nhiều mô
  hình hơn và tập kiểm tra nhỏ hơn. Báo cả phân phối, chấm lại trên dữ liệu mới.
* Nhãn sai là sai số của chính dụng cụ đo. Nhãn sai ngẫu nhiên tìm lại được bằng độ tự tin của mô
  hình, còn thiên lệch nhất quán thì không.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Đo chuẩn hiệu năng — khi lời hứa tối ưu gặp số đo](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch12-benchmarking/), dựng từ chương
[*Benchmarking*](https://mlsysbook.ai/vol1/benchmarking/benchmarking.html) của sách.

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.